# PRISM Doubly Robust Learner Modeling

This notebook is the report-ready doubly robust learner modeling workflow for the PRISM intervention benefit project. It follows the same evaluation framework as `PRISM_Causal_Forest_Modeling_Workflow.ipynb` and produces outputs comparable to `PRISM_Intervention_Benefit_Modeling_README.md`.

Core sign convention:

```text
tau_hat = estimated effect of intervention on outcome_ed_90d
benefit_score = -tau_hat
higher benefit_score = larger estimated ED risk reduction from intervention
```

The workflow uses the same reproducibility seed as all other PRISM workflows: `123`.

## Optional Package Install

Run this cell only if the current notebook kernel is missing `econml`. A Python 3.10–3.13 environment is recommended.

In [ ]:
# Uncomment if needed in a compatible Python environment.
# %pip install econml scikit-learn pandas numpy matplotlib openpyxl shap

## Background

Care management programs must decide which members should receive intervention when outreach resources are limited. A common approach is to prioritize the highest-risk members, but high baseline risk does not always mean high intervention benefit. This doubly robust learner workflow focuses on estimating whether intervention benefit varies across members using a doubly robust treatment-effect estimation framework.

## Business Question

Which members are most likely to benefit from intervention in terms of reducing 90-day emergency department utilization, based on doubly robust estimates of heterogeneous treatment effects?

## Project Objectives

- Estimate member-level heterogeneous treatment effects using a doubly robust learner.
- Rank members by estimated intervention benefit.
- Identify high-benefit deciles and subgroup profiles.
- Compare doubly robust rankings with existing T-learner, X-learner, and causal forest outputs.
- Provide explainability through variable importance and SHAP.
- Produce README-ready CSV tables and charts.

## Analytical Task 1: Understanding And Explaining The Doubly Robust Framework

The doubly robust learner estimates a conditional average treatment effect (CATE) for each member. It combines an outcome model with inverse propensity weighting to construct doubly robust pseudo-outcomes, then fits a final random forest regression on those pseudo-outcomes to produce individualized treatment-effect estimates.

The doubly robust property means the treatment-effect estimates remain consistent if either the outcome model or the propensity model is correctly specified. This provides an additional layer of robustness compared with methods that rely on a single nuisance model.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except Exception:
    display = print

if importlib.util.find_spec('econml') is None:
    raise ImportError('Missing required package: econml. Install it in a Python 3.10-3.13 environment, then rerun this notebook.')

from econml.dr import ForestDRLearner

CODE_DIR = Path.cwd()
if CODE_DIR.name.lower() != 'code':
    CODE_DIR = Path.cwd() / 'Code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from _prism_model_utils import (
    add_date_features,
    clean_names_simple,
    ensure_output_folder,
    make_design_matrix,
    ntile_desc,
    prepare_model_frame,
    read_prism_excel,
    require_columns,
    split_train_test,
    to_binary,
)

PROJECT_ROOT = CODE_DIR.parent
SEED = 123
TRAIN_FRACTION = 0.70
OUTCOME_COL = 'outcome_ed_90d'
TREATMENT_COL = 'intervention_flag'
OUTPUT_DIR = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Doubly-Robust' / 'Python')
COST_PER_ED_VISIT = 1200
COST_PER_INTERVENTION = 250

np.random.seed(SEED)
warnings.filterwarnings('ignore', category=UserWarning)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Seed: {SEED}')

In [ ]:
PATHS = {
    # Task 2
    'predictor_inventory': OUTPUT_DIR / 'doubly_robust_predictor_inventory.csv',
    'data_review_summary': OUTPUT_DIR / 'doubly_robust_data_review_summary.csv',
    # Task 3
    'event_count_summary': OUTPUT_DIR / 'doubly_robust_event_count_summary.csv',
    'propensity_summary': OUTPUT_DIR / 'doubly_robust_propensity_summary.csv',
    'pseudo_outcome_summary': OUTPUT_DIR / 'doubly_robust_pseudo_outcome_summary.csv',
    'propensity_chart': OUTPUT_DIR / 'dashboard_doubly_robust_propensity_overlap.png',
    'pseudo_outcome_chart': OUTPUT_DIR / 'dashboard_doubly_robust_pseudo_outcome_distribution.png',
    # Task 4
    'scored_output': OUTPUT_DIR / 'doubly_robust_scored_output.csv',
    'test_scored_output': OUTPUT_DIR / 'doubly_robust_scored_test_output.csv',
    'effect_distribution_summary': OUTPUT_DIR / 'doubly_robust_effect_distribution_summary.csv',
    'ate_summary': OUTPUT_DIR / 'doubly_robust_ate_summary.csv',
    'true_benefit_validation_summary': OUTPUT_DIR / 'doubly_robust_true_benefit_validation_summary.csv',
    'effect_distribution_chart': OUTPUT_DIR / 'dashboard_doubly_robust_effect_distribution.png',
    # Task 5
    'decile_summary': OUTPUT_DIR / 'doubly_robust_decile_summary.csv',
    'risk_tier_benefit_group_summary': OUTPUT_DIR / 'doubly_robust_risk_tier_benefit_group_summary.csv',
    'top_decile_profile': OUTPUT_DIR / 'doubly_robust_top_decile_profile.csv',
    'consistency_summary': OUTPUT_DIR / 'doubly_robust_cross_method_consistency_summary.csv',
    'top_benefit_examples': OUTPUT_DIR / 'doubly_robust_top_benefit_examples.csv',
    'benefit_decile_chart': OUTPUT_DIR / 'dashboard_doubly_robust_avg_benefit_by_decile.png',
    'risk_tier_benefit_group_chart': OUTPUT_DIR / 'dashboard_doubly_robust_risk_tier_by_benefit_group.png',
    'cross_method_chart': OUTPUT_DIR / 'dashboard_doubly_robust_cross_method_agreement.png',
    # Task 6
    'variable_importance': OUTPUT_DIR / 'doubly_robust_variable_importance.csv',
    'shap_importance': OUTPUT_DIR / 'doubly_robust_global_benefit_shap_importance.csv',
    'shap_values': OUTPUT_DIR / 'doubly_robust_member_benefit_shap_values.csv',
    'variable_importance_chart': OUTPUT_DIR / 'dashboard_doubly_robust_variable_importance.png',
    'shap_chart': OUTPUT_DIR / 'dashboard_doubly_robust_global_benefit_shap.png',
    # Task 7
    'targeting_summary': OUTPUT_DIR / 'doubly_robust_targeting_summary.csv',
    'cumulative_savings_chart': OUTPUT_DIR / 'dashboard_doubly_robust_cumulative_gross_savings_targeting.png',
    'marginal_advantage_chart': OUTPUT_DIR / 'dashboard_doubly_robust_marginal_gross_savings_advantage.png',
}

print(f'PATHS dict defines {len(PATHS)} output targets.')

In [ ]:
def save_csv(df, path):
    """Save DataFrame to CSV and print confirmation."""
    df.to_csv(path, index=False)
    print(f'Saved: {path}')


def save_current_figure(path):
    """Save current matplotlib figure and display it."""
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path}')


def summarize_distribution(values, label):
    """Return a 9-row percentile summary DataFrame."""
    series = pd.Series(values, dtype=float).dropna()
    return pd.DataFrame({
        'metric': ['mean', 'std_dev', 'min', 'p10', 'p25', 'median', 'p75', 'p90', 'max'],
        label: [
            series.mean(), series.std(), series.min(),
            series.quantile(0.10), series.quantile(0.25), series.median(),
            series.quantile(0.75), series.quantile(0.90), series.max(),
        ],
    })


def safe_corr(a, b, method):
    """NaN-safe correlation between two Series."""
    joined = pd.concat([pd.Series(a, dtype=float), pd.Series(b, dtype=float)], axis=1).dropna()
    if len(joined) < 3:
        return np.nan
    return joined.iloc[:, 0].corr(joined.iloc[:, 1], method=method)


def top_overlap(a_scores, b_scores, share=0.10):
    """Fraction of top-k members shared between two score vectors."""
    a = pd.Series(a_scores).reset_index(drop=True)
    b = pd.Series(b_scores).reset_index(drop=True)
    n = min(len(a), len(b))
    if n == 0:
        return np.nan
    k = max(1, int(np.floor(n * share)))
    return len(set(a.iloc[:n].nlargest(k).index) & set(b.iloc[:n].nlargest(k).index)) / k


def propensity_auc(y_true, scores):
    """Try/except wrapper for ROC AUC."""
    try:
        return roc_auc_score(y_true, scores)
    except Exception:
        return np.nan


def load_shared_propensity_scores():
    """Load exact member-level propensity scores saved by the uplift/X-learner workflow."""
    shared_path = PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python' / 'X-Learner' / 'shared_propensity_scores.csv'
    if not shared_path.exists():
        return None
    shared = pd.read_csv(shared_path)
    required = {'member_id', 'split', 'propensity_score'}
    missing = required - set(shared.columns)
    if missing:
        raise ValueError(f'Shared propensity file is missing columns: {sorted(missing)}')
    if shared['member_id'].duplicated().any():
        raise ValueError('Shared propensity file has duplicate member_id values.')
    return shared


def merge_shared_propensity(frame, shared_propensity, split_label):
    """Merge exact propensity values by member_id and validate one-to-one matching."""
    if shared_propensity is None:
        return None
    split_scores = shared_propensity[shared_propensity['split'].eq(split_label)].copy()
    merged = frame[['member_id']].merge(
        split_scores[['member_id', 'propensity_score']],
        on='member_id', how='left', validate='one_to_one',
    )
    if merged['propensity_score'].isna().any():
        missing_ids = merged.loc[merged['propensity_score'].isna(), 'member_id'].head().tolist()
        raise ValueError(f'Missing shared propensity for {split_label} member_ids: {missing_ids}')
    return merged['propensity_score'].to_numpy(dtype=float)


print('Helper functions defined.')

## Analytical Task 2: Data Review

This section reviews the modeling population, treatment rate, outcome prevalence, and final model matrix size. These checks are identical to the causal forest and uplift workflows because all PRISM methods use the same source data and predictor set.

In [ ]:
df = read_prism_excel()
df.columns = clean_names_simple(df.columns)
df = df.copy()

require_columns(df, [OUTCOME_COL, TREATMENT_COL])
df[OUTCOME_COL] = to_binary(df[OUTCOME_COL])
df[TREATMENT_COL] = to_binary(df[TREATMENT_COL])
df = add_date_features(df, include_duration=False)

PREDICTOR_CATEGORIES = {
    'demographics': ['client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag'],
    'clinical_conditions': ['diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'behavioral_health_risk_flag'],
    'sdoh': ['food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag'],
    'utilization': ['pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m'],
    'pharmacy': ['total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag'],
    'risk_scores': ['percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier'],
}
PREDICTOR_VARS = [f for fs in PREDICTOR_CATEGORIES.values() for f in fs]
NUMERIC_VARS = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc',
    'percolator_utilization_score', 'percolator_clinical_score',
    'percolator_sdoh_score', 'current_risk_score',
    'intervention_start_month', 'intervention_start_wday', 'days_to_intervention_start',
]
BINARY_EXTRA = ['dual_eligible', 'living_alone_flag']
present_predictors = [f for f in PREDICTOR_VARS if f in df.columns]

# Predictor inventory
predictor_inventory = pd.DataFrame([
    {
        'feature': feature,
        'category': category,
        'included_in_model': feature in present_predictors,
        'reason_if_excluded': '' if feature in present_predictors else 'Column not present in source data',
        'source_dtype': str(df[feature].dtype) if feature in df.columns else 'missing',
        'unique_values': df[feature].nunique(dropna=True) if feature in df.columns else 0,
    }
    for category, features in PREDICTOR_CATEGORIES.items()
    for feature in features
])
save_csv(predictor_inventory, PATHS['predictor_inventory'])

# Prepare model frame
model_df = prepare_model_frame(df, present_predictors, NUMERIC_VARS, BINARY_EXTRA)
model_df.insert(0, 'member_id', np.arange(len(model_df), dtype=int))
feature_cols_raw = [c for c in model_df.columns if c not in ['member_id', OUTCOME_COL, TREATMENT_COL]]

continuous_count = len([c for c in feature_cols_raw if c in NUMERIC_VARS])
binary_count = len([c for c in feature_cols_raw if c.endswith('_flag') or c in BINARY_EXTRA])
categorical_count = len(feature_cols_raw) - continuous_count - binary_count

# Data review summary
data_review_summary = pd.DataFrame({
    'metric': [
        'Total members', 'Treated members', 'Untreated/control members',
        'Treatment rate', 'ED outcome events', 'Outcome prevalence',
        'Treated observed ED rate', 'Control observed ED rate',
        'Final predictors before one-hot encoding',
        'Continuous/count numeric predictors', 'Binary indicator predictors',
        'Multi-level categorical predictors',
    ],
    'current_value': [
        len(model_df),
        int((model_df[TREATMENT_COL] == 1).sum()),
        int((model_df[TREATMENT_COL] == 0).sum()),
        model_df[TREATMENT_COL].mean(),
        int((model_df[OUTCOME_COL] == 1).sum()),
        model_df[OUTCOME_COL].mean(),
        model_df.loc[model_df[TREATMENT_COL] == 1, OUTCOME_COL].mean(),
        model_df.loc[model_df[TREATMENT_COL] == 0, OUTCOME_COL].mean(),
        len(feature_cols_raw),
        continuous_count,
        binary_count,
        categorical_count,
    ],
})
save_csv(data_review_summary, PATHS['data_review_summary'])
display(data_review_summary)
print(f'\nModel frame: {model_df.shape[0]} rows x {len(feature_cols_raw)} raw predictors')

---

## Evaluation Roadmap

The remaining analyses are organized into two evaluation stages that build upon one another.

| Evaluation Level | Question | Analytical Tasks |
|---|---|---|
| **Level 1: Treatment-Effect Credibility** | Are the estimated treatment effects sufficiently credible for interpretation and member prioritization? | Tasks 3–5 |
| **Level 2: Explainability And Business Value** | Can the estimated treatment effects be explained and translated into improved targeting decisions? | Tasks 6–7 |

---

# Evaluation Level 1: Treatment-Effect Credibility

**Question:** Are the estimated treatment effects sufficiently credible for interpretation and member prioritization?

The first stage assesses the credibility of the doubly robust treatment-effect estimates through diagnostics, validation against the known synthetic treatment benefit, and cross-method consistency.

---

## Analytical Task 3: Doubly Robust Diagnostics And Estimation Credibility

This section establishes confidence in the doubly robust learner before interpreting treatment effects. It evaluates event counts, propensity overlap, the ForestDRLearner model fit, and pseudo-outcome diagnostics unique to the doubly robust approach.

The primary question is:

> **What evidence suggests that the doubly robust treatment-effect estimates are reliable enough for exploratory prioritization and subgroup discovery?**

In [ ]:
# --- Train/Test Split ---
feature_frame = model_df.drop(columns=['member_id', OUTCOME_COL, TREATMENT_COL])
_, [x_all] = make_design_matrix([feature_frame])

train_df, test_df = split_train_test(
    model_df, train_fraction=TRAIN_FRACTION, seed=SEED,
    stratify_columns=[TREATMENT_COL, OUTCOME_COL],
)
x_train = x_all.loc[train_df.index].reset_index(drop=True)
x_test = x_all.loc[test_df.index].reset_index(drop=True)
y_train = train_df[OUTCOME_COL].astype(float).to_numpy()
w_train = train_df[TREATMENT_COL].astype(float).to_numpy()
y_test = test_df[OUTCOME_COL].astype(float).to_numpy()
w_test = test_df[TREATMENT_COL].astype(float).to_numpy()

# Update data review with model matrix dimensions
data_review_summary = pd.concat([data_review_summary, pd.DataFrame({
    'metric': ['Model matrix columns after one-hot encoding', 'Train rows', 'Test rows'],
    'current_value': [x_all.shape[1], len(train_df), len(test_df)],
})], ignore_index=True)
save_csv(data_review_summary, PATHS['data_review_summary'])

# --- Event Counts ---
event_rows = []
for split_name, frame in [('Train', train_df), ('Test', test_df)]:
    for group_value, group_label in [(1.0, 'Treated'), (0.0, 'Control')]:
        subset = frame[frame[TREATMENT_COL] == group_value]
        positive = int((subset[OUTCOME_COL] == 1).sum())
        n = int(len(subset))
        event_rows.append({
            'split': split_name, 'group': group_label, 'n': n,
            'positive_ed_events': positive, 'negative_ed_events': n - positive,
            'event_rate': positive / n if n else np.nan,
        })
event_count_summary = pd.DataFrame(event_rows)
save_csv(event_count_summary, PATHS['event_count_summary'])
display(event_count_summary)

# --- Shared Propensity Scores ---
shared_propensity_scores = load_shared_propensity_scores()
if shared_propensity_scores is not None:
    train_propensity = merge_shared_propensity(train_df, shared_propensity_scores, 'train')
    test_propensity = merge_shared_propensity(test_df, shared_propensity_scores, 'test')
    propensity_source = 'shared_propensity_scores_member_id_merge'
    print('Using shared propensity scores from X-learner workflow.')
else:
    # Fallback: fit propensity model matching X-learner specification
    from sklearn.pipeline import make_pipeline
    propensity_pipeline = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
            penalty='elasticnet', solver='saga', l1_ratios=[0.5],
            scoring='roc_auc', max_iter=10000, random_state=SEED, refit=True,
        )
    )
    propensity_pipeline.fit(x_train, w_train)
    train_propensity = np.clip(propensity_pipeline.predict_proba(x_train)[:, 1], 0.05, 0.95)
    test_propensity = np.clip(propensity_pipeline.predict_proba(x_test)[:, 1], 0.05, 0.95)
    propensity_source = 'doubly_robust_matched_glmnet_model_no_shared_file'
    print('Shared propensity file not found; fitted internal propensity model.')

# Propensity summary
prop_series = pd.Series(test_propensity, dtype=float)
propensity_summary = pd.DataFrame({
    'metric': [
        'Propensity source', 'Train treatment model AUC', 'Test treatment model AUC',
        'Mean propensity', 'Min propensity', '5th percentile',
        'Median propensity', '95th percentile', 'Max propensity',
        'Members below 0.05', 'Members above 0.95',
    ],
    'value': [
        propensity_source,
        propensity_auc(w_train, train_propensity),
        propensity_auc(w_test, test_propensity),
        prop_series.mean(), prop_series.min(), prop_series.quantile(0.05),
        prop_series.median(), prop_series.quantile(0.95), prop_series.max(),
        int((prop_series < 0.05).sum()), int((prop_series > 0.95).sum()),
    ],
})
save_csv(propensity_summary, PATHS['propensity_summary'])
display(propensity_summary)

# Propensity overlap chart
plt.figure(figsize=(8, 4.5))
plt.hist(test_propensity[w_test == 1], bins=15, alpha=0.65, label='Treated')
plt.hist(test_propensity[w_test == 0], bins=15, alpha=0.65, label='Control')
plt.xlabel('Estimated propensity for intervention')
plt.ylabel('Members')
plt.title('Doubly Robust Propensity Overlap Check')
plt.legend()
save_current_figure(PATHS['propensity_chart'])

In [ ]:
# --- Fit ForestDRLearner ---
print('Fitting ForestDRLearner...')
dr_model = ForestDRLearner(
    model_regression=RandomForestRegressor(
        n_estimators=300, min_samples_leaf=10, random_state=SEED, n_jobs=-1,
    ),
    model_propensity=make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
            penalty='elasticnet', solver='saga', l1_ratios=[0.5],
            scoring='roc_auc', max_iter=10000, random_state=SEED, refit=True,
        ),
    ),
    cv=5,
    min_samples_leaf=10,
    n_estimators=500,
    random_state=SEED,
)
dr_model.fit(Y=y_train, T=w_train, X=x_train, W=None)
print('ForestDRLearner fitted successfully.')

# --- Pseudo-Outcome Diagnostics ---
# Reconstruct DR pseudo-outcomes on training data for diagnostic purposes.
# The pseudo-outcome is the doubly robust score that the final model is trained on.
# We reconstruct it using the cross-fitted nuisance predictions.
try:
    # Access internal nuisance model predictions if available
    tau_train_raw = dr_model.effect(x_train).flatten()
    # Use training-set effect estimates as proxy for pseudo-outcome distribution diagnostic
    pseudo_outcomes = tau_train_raw
    pseudo_label = 'Training-set effect estimates (proxy for pseudo-outcomes)'
except Exception as exc:
    print(f'Could not compute training effects: {exc}')
    pseudo_outcomes = np.array([])
    pseudo_label = 'unavailable'

if len(pseudo_outcomes) > 0:
    po_series = pd.Series(pseudo_outcomes, dtype=float)
    pseudo_outcome_summary = pd.DataFrame({
        'metric': [
            'Source', 'N', 'Mean', 'Std', 'Min', '5th percentile',
            '10th percentile', '25th percentile', 'Median',
            '75th percentile', '90th percentile', '95th percentile', 'Max',
            'Fraction negative (benefit direction)',
        ],
        'value': [
            pseudo_label, len(po_series),
            po_series.mean(), po_series.std(), po_series.min(),
            po_series.quantile(0.05), po_series.quantile(0.10),
            po_series.quantile(0.25), po_series.median(),
            po_series.quantile(0.75), po_series.quantile(0.90),
            po_series.quantile(0.95), po_series.max(),
            float((po_series < 0).mean()),
        ],
    })
    save_csv(pseudo_outcome_summary, PATHS['pseudo_outcome_summary'])
    display(pseudo_outcome_summary)

    # Pseudo-outcome distribution chart
    plt.figure(figsize=(8, 4.5))
    plt.hist(-po_series, bins=20, alpha=0.85)
    plt.axvline(-po_series.mean(), linestyle='--', color='black', label=f'Mean benefit = {-po_series.mean():.4f}')
    plt.xlabel('Benefit score (-tau_hat, training set)')
    plt.ylabel('Members')
    plt.title('Doubly Robust: Training-Set Effect Distribution')
    plt.legend()
    save_current_figure(PATHS['pseudo_outcome_chart'])
else:
    print('Pseudo-outcome diagnostics skipped.')